In [ ]:
#@title Setup

import os
import gin
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from IPython.display import display

from smart_buildings.smart_control.environment import environment, hybrid_action_environment
from smart_buildings.smart_control.proto import smart_control_building_pb2
from smart_buildings.smart_control.proto import smart_control_reward_pb2
from smart_buildings.smart_control.reward import setpoint_energy_carbon_regret
from smart_buildings.smart_control.reward import electricity_energy_cost
from smart_buildings.smart_control.reward import natural_gas_energy_cost
from smart_buildings.smart_control.simulator import base_convection_simulator
from smart_buildings.smart_control.simulator import building
from smart_buildings.smart_control.simulator import hvac
from smart_buildings.smart_control.simulator import hvac_floorplan_based
from smart_buildings.smart_control.simulator import stochastic_convection_simulator
from smart_buildings.smart_control.simulator import tf_simulator
from smart_buildings.smart_control.simulator import randomized_arrival_departure_occupancy
from smart_buildings.smart_control.simulator import simulator_building
from smart_buildings.smart_control.utils import building_renderer
from smart_buildings.smart_control.utils import conversion_utils
from smart_buildings.smart_control.utils import real_building_temperature_array_generator as temp_array
from smart_buildings.smart_control.utils import environment_utils
from smart_buildings.smart_control.utils import observation_normalizer

floor_plan_path = os.path.join(os.dirname(__file__), "..", "..", "configs", "resources", "sb1", "double_resolution_zone_1_2.npy") #@param {type:"string"}

def plot_hvac_data_with_timestamps(data_tuples):
    """
    Parses and plots HVAC data from tuples, including explicit timestamps.

    Args:
        data_tuples: List of (temps, setpoints, outside_air, pd_timestamp)
                     temps: list of k floats
                     setpoints: list of [low, high]
                     outside_air: float scalar
                     pd_timestamp: pandas.Timestamp object
    """

    if not data_tuples:
        print("No data to plot.")
        return

    # 1. Parse Data
    # Initialize lists to hold the components
    timestamps = []
    oa_data = []
    sp_low_data = []
    sp_high_data = []

    # Check the consistency of 'k' (number of temperature sensors)
    k = len(data_tuples[0][0]) if data_tuples else 0
    if k == 0:
        print("Temps list is empty in the first tuple.")
        return

    # Pre-allocate list of lists for the 'k' temp sensors
    temp_traces = [[] for _ in range(k)]

    # Unpack the 4 elements from the tuple
    for t_list, sp_list, oa_val, timestamp in data_tuples:
        timestamps.append(timestamp)
        oa_data.append(oa_val)
        sp_low_data.append(sp_list[0])
        sp_high_data.append(sp_list[1])

        # Distribute the k temps into their respective trace lists
        for i in range(k):
            temp_traces[i].append(t_list[i])

    # Convert timestamps list to datetime objects for matplotlib
    mpl_timestamps = [ts.to_pydatetime() for ts in timestamps]

    fig, ax = plt.subplots(figsize=(12, 6))

    # Plot Setpoint Range (Green)
    ax.fill_between(mpl_timestamps, sp_low_data, sp_high_data,
                    color='green', alpha=0.3, label='Setpoint Range')

    # Plot Outside Air (Blue)
    ax.plot(mpl_timestamps, oa_data, color='blue', linewidth=2, label='Outside Air')

    # Plot Temps (Yellow)
    for i, trace in enumerate(temp_traces):
        label = 'Temps' if i == 0 else "_nolegend_" # Only label the first one
        # Use a slightly transparent color if k is large
        ax.plot(mpl_timestamps, trace, color='#FFD700', linewidth=1.5, alpha=0.7, label=label)

    ax.set_ylabel('Temperature (°C)')
    ax.set_xlabel('Time')
    ax.set_title(f'HVAC System Monitor ({k} Zone Sensors)')

    # Format X-axis to handle dates nicely
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d\n%H:%M'))
    fig.autofmt_xdate()

    ax.grid(True, linestyle='--', alpha=0.6)
    ax.legend(loc='upper left')

    plt.tight_layout()
    plt.show()


In [ ]:
    #@title Config

def get_config(high_temp,low_temp):
  gin_config_contents=f"""
    ########a#######################################################
    # Low occupancy, constant setpoint
    ###############################################################

    ##########################
    ### WEATHER CONTROLLER
    ##########################

    # ORIGINAL:
    convection_coefficient = 100.0
    ambient_high_temp = {high_temp}#280#305 # K
    ambient_low_temp =  {low_temp}#280#305 # K

    sim/WeatherController:
      default_low_temp = %ambient_low_temp
      default_high_temp = %ambient_high_temp
      convection_coefficient = %convection_coefficient

    weather_controller = @sim/WeatherController()




    ##########################
    ### BUILDING
    ##########################

    initial_temp = 294.0
    control_volume_cm = 10 #20
    floor_height_cm = 300.0

    #floor_plan_filepath = "/cns/oz-d/home/smart_buildings/control/floor_plan/floorplans_1_2.npy"
    #floor_plan_filepath = "/cns/oz-d/home/smart_buildings/control/floor_plan/double_uniform_zone_1_2.npy"
    #floor_plan_filepath = "/third_party/py/smart_buildings/smart_control/configs/resources/us_mtv_1055/double_resolution_zone_1_2.npy"
    floor_plan_filepath = "{floor_plan_path}"
    zone_map_filepath = "{floor_plan_path}"

    exterior_cv_conductivity = 5.5 #0.05 # Biggest cause of divergence
    exterior_cv_density = 1.0
    exterior_cv_heat_capacity = 700.0

    interior_wall_cv_conductivity = 50.0
    interior_wall_cv_density = 1.0
    interior_wall_cv_heat_capacity = 700.0

    interior_cv_conductivity =  50.0
    interior_cv_density = 0.1#1.0
    interior_cv_heat_capacity = 700

    inside_air_properties/MaterialProperties:
      conductivity = %interior_cv_conductivity
      heat_capacity = %interior_cv_heat_capacity
      density = %interior_cv_density

    inside_wall_properties/MaterialProperties:
      conductivity = %interior_wall_cv_conductivity
      heat_capacity = %interior_wall_cv_heat_capacity
      density = %interior_wall_cv_density

    building_exterior_properties/MaterialProperties:
      conductivity = %exterior_cv_conductivity
      heat_capacity = %exterior_cv_heat_capacity
      density = %exterior_cv_density

    sim/FloorPlanBasedBuilding:
      cv_size_cm = %control_volume_cm
      floor_height_cm = %floor_height_cm
      initial_temp  = %initial_temp
      inside_air_properties = @inside_air_properties/MaterialProperties()
      inside_wall_properties = @inside_wall_properties/MaterialProperties()
      building_exterior_properties = @building_exterior_properties/MaterialProperties()
      floor_plan_filepath = %floor_plan_filepath
      zone_map_filepath = %zone_map_filepath
      #save_debugging_images = False
      #convection_simulator = @StochasticConvectionSimulator()


    ##########################
    ### SCHEDULE ...
    ### https://source.corp.google.com/piper///depot/google3/corp/ml/smart_buildings/smart_control/configs/real_building/us_mtv_1055/base_config.gin;l=7?q=%23%20HVAC%20heating%2Fcooling%20schedule&ct=os&sq=package:piper%20file:%2F%2Fdepot%2Fgoogle3%20-file:google3%2Fexperimental
    ##########################

    morning_start_hour =  6
    evening_start_hour = 19
    heating_setpoint_day = 294
    cooling_setpoint_day = 297
    heating_setpoint_night = 289
    cooling_setpoint_night = 298
    time_zone="US/Pacific"

    hvac/SetpointSchedule:
      morning_start_hour = %morning_start_hour
      evening_start_hour = %evening_start_hour
      comfort_temp_window = (%heating_setpoint_day, %cooling_setpoint_day)
      eco_temp_window = (%heating_setpoint_night, %cooling_setpoint_night)
      time_zone = %time_zone



    ##########################
    ### HVAC
    ##########################

    water_pump_differential_head =  6.0
    water_pump_efficiency = 0.98
    reheat_water_setpoint = 360.0
    boiler_heating_rate = 0.5 # K / min
    boiler_cooling_rate = 0.1 # K / min

    fan_static_pressure = 10000.0
    fan_efficiency = 0.9

    air_handler_heating_setpoint = 285.0
    air_handler_cooling_setpoint = 298.0
    air_handler_recirculation_ratio = 0.3

    vav_max_air_flowrate = 2.0#0.035
    vav_reheat_water_flowrate =  0.03

    # hvac/AirHandler:
    #   recirculation = %air_handler_recirculation_ratio
    #   heating_air_temp_setpoint = %air_handler_heating_setpoint
    #   cooling_air_temp_setpoint = %air_handler_cooling_setpoint
    #   fan_static_pressure = %fan_static_pressure
    #   fan_efficiency = %fan_efficiency
    #   max_air_flow_rate = 8.67
    #   sim_weather_controller = %weather_controller

    floor_1_rooms = [
      'zone_id_1', 'zone_id_2', 'zone_id_3', 'zone_id_4', 'zone_id_5', 'zone_id_6', 'zone_id_7', 'zone_id_8', 'zone_id_9', 'zone_id_10',
      'zone_id_11', 'zone_id_12', 'zone_id_13', 'zone_id_14', 'zone_id_15', 'zone_id_16', 'zone_id_17', 'zone_id_18', 'zone_id_19', 'zone_id_20',
      'zone_id_21', 'zone_id_22', 'zone_id_23', 'zone_id_24', 'zone_id_25', 'zone_id_26', 'zone_id_27', 'zone_id_28', 'zone_id_29', 'zone_id_30',
      'zone_id_31', 'zone_id_32', 'zone_id_33', 'zone_id_34', 'zone_id_35', 'zone_id_36', 'zone_id_37', 'zone_id_38', 'zone_id_39', 'zone_id_40',
      'zone_id_41', 'zone_id_42', 'zone_id_43', 'zone_id_44', 'zone_id_45', 'zone_id_46', 'zone_id_47', 'zone_id_48', 'zone_id_49', 'zone_id_50',
      'zone_id_51', 'zone_id_52', 'zone_id_53'
    ]

    floor_2_rooms = [
      'zone_id_54', 'zone_id_55', 'zone_id_56', 'zone_id_57', 'zone_id_58', 'zone_id_59', 'zone_id_60', 'zone_id_61', 'zone_id_62', 'zone_id_63',
      'zone_id_64', 'zone_id_65', 'zone_id_66', 'zone_id_67', 'zone_id_68', 'zone_id_69', 'zone_id_70', 'zone_id_71', 'zone_id_72', 'zone_id_73',
      'zone_id_74', 'zone_id_75', 'zone_id_76', 'zone_id_77', 'zone_id_78', 'zone_id_79', 'zone_id_80', 'zone_id_81', 'zone_id_82', 'zone_id_83',
      'zone_id_84', 'zone_id_85', 'zone_id_86', 'zone_id_87', 'zone_id_88', 'zone_id_89', 'zone_id_90', 'zone_id_91', 'zone_id_92', 'zone_id_93',
      'zone_id_94', 'zone_id_95', 'zone_id_96', 'zone_id_97', 'zone_id_98', 'zone_id_99', 'zone_id_100', 'zone_id_101', 'zone_id_102', 'zone_id_103',
      'zone_id_104', 'zone_id_105', 'zone_id_106', 'zone_id_107', 'zone_id_108', 'zone_id_109', 'zone_id_110', 'zone_id_111', 'zone_id_112', 'zone_id_113',
      'zone_id_114', 'zone_id_115', 'zone_id_116', 'zone_id_117', 'zone_id_118', 'zone_id_119', 'zone_id_120', 'zone_id_121', 'zone_id_122', 'zone_id_123',
      'zone_id_124', 'zone_id_125', 'zone_id_126'
    ]

    ahu_1/AirHandler:
      recirculation = %air_handler_recirculation_ratio
      heating_air_temp_setpoint = %air_handler_heating_setpoint
      cooling_air_temp_setpoint = %air_handler_cooling_setpoint
      fan_static_pressure = %fan_static_pressure
      fan_efficiency = %fan_efficiency
      max_air_flow_rate = 8.67
      sim_weather_controller = %weather_controller
      device_id = 'ahu_1'

    ahu_2/AirHandler:
      recirculation = %air_handler_recirculation_ratio
      heating_air_temp_setpoint = %air_handler_heating_setpoint
      cooling_air_temp_setpoint = %air_handler_cooling_setpoint
      fan_static_pressure = %fan_static_pressure
      fan_efficiency = %fan_efficiency
      max_air_flow_rate = 8.67
      sim_weather_controller = %weather_controller
      device_id = 'ahu_2'

    hvac/AirHandlerSystem:
      ahus =  {{
        @ahu_1/AirHandler(): %floor_1_rooms,
        @ahu_2/AirHandler(): %floor_2_rooms,
      }}
      device_id = 'ahu'

    hvac/WaterPump:
      water_pump_differential_head = %water_pump_differential_head
      water_pump_efficiency = %water_pump_efficiency

    hvac/Boiler:
      reheat_water_setpoint = %reheat_water_setpoint
      heating_rate = %boiler_heating_rate
      cooling_rate = %boiler_cooling_rate

    hvac/HotWaterSystem:
      pump = @hvac/WaterPump()
      boiler = @hvac/Boiler()
      device_id = 'hws'


    sim/FloorPlanBasedHvac:
      air_handler = @hvac/AirHandlerSystem()
      hot_water_system = @hvac/HotWaterSystem()
      schedule = @hvac/SetpointSchedule()
      vav_max_air_flow_rate = %vav_max_air_flowrate
      vav_reheat_max_water_flow_factor = %vav_reheat_water_flowrate


    ##########################
    ### SIMULATOR
    ##########################
    # shuffle parameters
    StochasticConvectionSimulator.p = 1.0
    StochasticConvectionSimulator.distance = -1
    StochasticConvectionSimulator.seed = 5

    # Finite difference settings.
    time_step_sec =  300
    convergence_threshold = 0.01
    iteration_limit = 100
    iteration_warning = 20
    start_timestamp = '2023-07-10 19:00' # '2021-04-01 00:00'

    sim/to_timestamp.date_str = %start_timestamp

    #sim_building/SimulatorFlexibleGeometries:
    sim_building/TFSimulator:
      building = @sim/FloorPlanBasedBuilding()
      hvac  = @sim/FloorPlanBasedHvac()
      weather_controller = %weather_controller
      time_step_sec = %time_step_sec
      convergence_threshold = %convergence_threshold
      iteration_limit = %iteration_limit
      iteration_warning = %iteration_warning
      start_timestamp = @sim/to_timestamp()


    work_occupancy = 1
    nonwork_occupancy = 0.1
    # occupancy_start_time = '07:00:00'
    # occupancy_end_time = '17:00:00'
    occupancy_start/local_time.time_str =  %occupancy_start_time
    occupancy_end/local_time.time_str = %occupancy_end_time


    randomized_occupancy/RandomizedArrivalDepartureOccupancy:
      zone_assignment = %work_occupancy
      earliest_expected_arrival_hour = 3
      latest_expected_arrival_hour = 12
      earliest_expected_departure_hour = 13
      latest_expected_departure_hour = 23
      time_step_sec = %time_step_sec


    #SimulatorBuilding.simulator = @sim_building/SimulatorFlexibleGeometries()
    SimulatorBuilding.simulator = @sim_building/TFSimulator()
    # SimulatorBuilding.occupancy = @step_function_occupancy/StepFunctionOccupancy()
    SimulatorBuilding.occupancy = @randomized_occupancy/RandomizedArrivalDepartureOccupancy()


    ##########################
    ### REWARDS
    ##########################

    productivity_personhour_usd = 300.00

    productivity_midpoint_delta_temp =  1.5
    decay_stiffness =  4.3

    electricity_weight =  1.0
    carbon_weight =  1.0

    reward_normalizer_shift = 0.0
    reward_normalizer_scale = 450.0

    max_productivity_personhour_usd = 300.00
    min_productivity_personhour_usd = 100.00
    productivity_midpoint_delta =  0.5
    productivity_decay_stiffness =  4.3

    max_electricity_rate=160000
    max_natural_gas_rate=400000

    productivity_weight=0.2
    energy_cost_weight=0.4
    carbon_emission_weight=0.4

    SetpointEnergyCarbonRegretFunction.max_productivity_personhour_usd = %max_productivity_personhour_usd
    SetpointEnergyCarbonRegretFunction.min_productivity_personhour_usd = %min_productivity_personhour_usd
    SetpointEnergyCarbonRegretFunction.max_electricity_rate = %max_electricity_rate
    SetpointEnergyCarbonRegretFunction.max_natural_gas_rate = %max_natural_gas_rate
    SetpointEnergyCarbonRegretFunction.productivity_decay_stiffness = %productivity_decay_stiffness
    SetpointEnergyCarbonRegretFunction.productivity_midpoint_delta = %productivity_midpoint_delta
    SetpointEnergyCarbonRegretFunction.electricity_energy_cost = @ElectricityEnergyCost()
    SetpointEnergyCarbonRegretFunction.natural_gas_energy_cost = @NaturalGasEnergyCost()
    SetpointEnergyCarbonRegretFunction.productivity_weight = %productivity_weight
    SetpointEnergyCarbonRegretFunction.energy_cost_weight= %energy_cost_weight
    SetpointEnergyCarbonRegretFunction.carbon_emission_weight = %carbon_emission_weight


    ##########################
    ### ACTIONS
    ##########################

    # Action Normalization Parameters -> edited to match real building: https://source.corp.google.com/piper///depot/google3/corp/ml/smart_buildings/smart_control/configs/real_building/us_mtv_1055/base_config.gin;rcl=520286766;l=89
    supply_water_bounded_action_normalizer/set_action_normalization_constants.min_normalized_value = -1.
    supply_water_bounded_action_normalizer/set_action_normalization_constants.max_normalized_value = 1.0
    supply_water_bounded_action_normalizer/set_action_normalization_constants.min_native_value = 310 #300.0
    supply_water_bounded_action_normalizer/set_action_normalization_constants.max_native_value = 350.0

    supply_air_heating_temperature_setpoint/set_action_normalization_constants.min_normalized_value = -1.
    supply_air_heating_temperature_setpoint/set_action_normalization_constants.max_normalized_value = 1.
    supply_air_heating_temperature_setpoint/set_action_normalization_constants.min_native_value = 285 #275.0
    supply_air_heating_temperature_setpoint/set_action_normalization_constants.max_native_value = 295.0

    supply_air_temperature_setpoint/set_action_normalization_constants.min_normalized_value = -1.
    supply_air_temperature_setpoint/set_action_normalization_constants.max_normalized_value = 1.
    supply_air_temperature_setpoint/set_action_normalization_constants.min_native_value = 285 #275.0
    supply_air_temperature_setpoint/set_action_normalization_constants.max_native_value = 305.0

    differential_pressure_setpoint/set_action_normalization_constants.min_normalized_value = -1.
    differential_pressure_setpoint/set_action_normalization_constants.max_normalized_value = 1.
    differential_pressure_setpoint/set_action_normalization_constants.min_native_value = 0 #275.0
    differential_pressure_setpoint/set_action_normalization_constants.max_native_value = 20.0

    static_pressure_setpoint/set_action_normalization_constants.min_normalized_value = -1.
    static_pressure_setpoint/set_action_normalization_constants.max_normalized_value = 1.
    static_pressure_setpoint/set_action_normalization_constants.min_native_value = 0 #275.0
    static_pressure_setpoint/set_action_normalization_constants.max_native_value = 20000.0

    run_command/set_action_normalization_constants.min_normalized_value = -1.
    run_command/set_action_normalization_constants.max_normalized_value = 1.
    run_command/set_action_normalization_constants.min_native_value = 0.0
    run_command/set_action_normalization_constants.max_native_value = 1.0

    action_normalizer_map = {{
        'supply_water_setpoint': @supply_water_bounded_action_normalizer/set_action_normalization_constants(),
        'differential_pressure': @differential_pressure_setpoint/set_action_normalization_constants(),
        'ahu_1_supply_air_temperature_setpoint': @supply_air_temperature_setpoint/set_action_normalization_constants(),
        'ahu_1_static_pressure_setpoint': @static_pressure_setpoint/set_action_normalization_constants(),
        'ahu_2_supply_air_temperature_setpoint': @supply_air_temperature_setpoint/set_action_normalization_constants(),
        'ahu_2_static_pressure_setpoint': @static_pressure_setpoint/set_action_normalization_constants(),
        'supervisor_run_command': @run_command/set_action_normalization_constants(),
        'ahu_1_supervisor_run_command': @run_command/set_action_normalization_constants(),
        'ahu_2_supervisor_run_command': @run_command/set_action_normalization_constants(),
    }}
    ActionConfig:
        action_normalizers = %action_normalizer_map

    default_actions = {{
        'supply_water_setpoint': 340.0,
        'differential_pressure': 20.0,
        'ahu_1_supply_air_temperature_setpoint': 293.0,
        'ahu_1_static_pressure_setpoint': 20000.0,
        'ahu_2_supply_air_temperature_setpoint': 293.0,
        'ahu_2_static_pressure_setpoint': 20000.0,
        'supervisor_run_command': 1.0,
        'ahu_1_supervisor_run_command': 1.0,
        'ahu_2_supervisor_run_command': 1.0,

    }}


    ##########################
    ### OBSERVATIONS
    ##########################

    temperature_observation_normalizer/set_observation_normalization_constants.field_id = 'temperature'
    temperature_observation_normalizer/set_observation_normalization_constants.sample_mean =  310.0
    temperature_observation_normalizer/set_observation_normalization_constants.sample_variance =  2500.0

    supply_water_setpoint_observation_normalizer/set_observation_normalization_constants.field_id = 'supply_water_setpoint'
    supply_water_setpoint_observation_normalizer/set_observation_normalization_constants.sample_mean =  310.0
    supply_water_setpoint_observation_normalizer/set_observation_normalization_constants.sample_variance =  2500.0

    air_flowrate_observation_normalizer/set_observation_normalization_constants.field_id = 'air_flowrate'
    air_flowrate_observation_normalizer/set_observation_normalization_constants.sample_mean =  0.5
    air_flowrate_observation_normalizer/set_observation_normalization_constants.sample_variance =  4.0

    differential_pressure_observation_normalizer/set_observation_normalization_constants.field_id = 'differential_pressure'
    differential_pressure_observation_normalizer/set_observation_normalization_constants.sample_mean =  10000.0
    differential_pressure_observation_normalizer/set_observation_normalization_constants.sample_variance =  100000.0

    percentage_observation_normalizer/set_observation_normalization_constants.field_id = 'percentage'
    percentage_observation_normalizer/set_observation_normalization_constants.sample_mean =  0.50
    percentage_observation_normalizer/set_observation_normalization_constants.sample_variance =  1.0

    request_count_observation_normalizer/set_observation_normalization_constants.field_id = 'request_count'
    request_count_observation_normalizer/set_observation_normalization_constants.sample_mean =  100.0
    request_count_observation_normalizer/set_observation_normalization_constants.sample_variance =  25.0

    # observation_normalizer_map = {{
    #     'temperature' : @temperature_observation_normalizer/set_observation_normalization_constants(),
    #     'supply_water_setpoint' : @supply_water_setpoint_observation_normalizer/set_observation_normalization_constants(),
    #     'air_flowrate': @air_flowrate_observation_normalizer/set_observation_normalization_constants(),
    #     'differential_pressure': @differential_pressure_observation_normalizer/set_observation_normalization_constants(),
    #     'percentage': @percentage_observation_normalizer/set_observation_normalization_constants(),
    #     'request_count': @request_count_observation_normalizer/set_observation_normalization_constants(),
    # }}
    # measurement 0 building_air_static_pressure_sensor
    building_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.field_id = 'building_air_static_pressure_sensor'
    building_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.sample_mean = 3.779228
    building_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.sample_variance = 14.599437

    # measurement 1 building_air_static_pressure_setpoint
    building_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'building_air_static_pressure_setpoint'
    building_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 7.472401
    building_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 0.000000

    # measurement 2 cooling_percentage_command
    cooling_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'cooling_percentage_command'
    cooling_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 9.658281
    cooling_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 295.833612

    # measurement 3 differential_pressure_sensor
    differential_pressure_sensor_normalizer/set_observation_normalization_constants.field_id = 'differential_pressure_sensor'
    differential_pressure_sensor_normalizer/set_observation_normalization_constants.sample_mean = 31611.814379
    differential_pressure_sensor_normalizer/set_observation_normalization_constants.sample_variance = 1844378631.487996

    # measurement 4 differential_pressure_setpoint
    differential_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'differential_pressure_setpoint'
    differential_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 83810.269540
    differential_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 14889040.603647

    # measurement 5 discharge_air_temperature_sensor
    discharge_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'discharge_air_temperature_sensor'
    discharge_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 69.889025
    discharge_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 541.455462

    # measurement 6 discharge_air_temperature_setpoint
    discharge_air_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'discharge_air_temperature_setpoint'
    discharge_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 57.665244
    discharge_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 97.254479

    # measurement 7 exhaust_air_damper_percentage_command
    exhaust_air_damper_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'exhaust_air_damper_percentage_command'
    exhaust_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 25.000000
    exhaust_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 0.000000

    # measurement 8 exhaust_air_damper_percentage_sensor
    exhaust_air_damper_percentage_sensor_normalizer/set_observation_normalization_constants.field_id = 'exhaust_air_damper_percentage_sensor'
    exhaust_air_damper_percentage_sensor_normalizer/set_observation_normalization_constants.sample_mean = 10.680755
    exhaust_air_damper_percentage_sensor_normalizer/set_observation_normalization_constants.sample_variance = 539.207818

    # measurement 9 exhaust_fan_speed_frequency_sensor
    exhaust_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.field_id = 'exhaust_fan_speed_frequency_sensor'
    exhaust_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_mean = 4.273057
    exhaust_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_variance = 138.559759

    # measurement 10 exhaust_fan_speed_percentage_command
    exhaust_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'exhaust_fan_speed_percentage_command'
    exhaust_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 7.121761
    exhaust_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 384.888218

    # measurement 11 heating_water_valve_percentage_command
    heating_water_valve_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'heating_water_valve_percentage_command'
    heating_water_valve_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 3.105189
    heating_water_valve_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 202.006249

    # measurement 12 mixed_air_temperature_sensor
    mixed_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'mixed_air_temperature_sensor'
    mixed_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 293.718710
    mixed_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 12.517696

    # measurement 13 mixed_air_temperature_setpoint
    mixed_air_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'mixed_air_temperature_setpoint'
    mixed_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 288.218302
    mixed_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 3.186768

    # measurement 14 outside_air_damper_percentage_command
    outside_air_damper_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'outside_air_damper_percentage_command'
    outside_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 34.504101
    outside_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 2053.149002

    # measurement 15 outside_air_dewpoint_temperature_sensor
    outside_air_dewpoint_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_dewpoint_temperature_sensor'
    outside_air_dewpoint_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 285.774428
    outside_air_dewpoint_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 2.504610

    # measurement 16 outside_air_flowrate_sensor
    outside_air_flowrate_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_flowrate_sensor'
    outside_air_flowrate_sensor_normalizer/set_observation_normalization_constants.sample_mean = 3.701930
    outside_air_flowrate_sensor_normalizer/set_observation_normalization_constants.sample_variance = 20.300565

    # measurement 17 outside_air_flowrate_setpoint
    outside_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.field_id = 'outside_air_flowrate_setpoint'
    outside_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 8.730134
    outside_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 0.240364

    # measurement 18 outside_air_relative_humidity_sensor
    outside_air_relative_humidity_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_relative_humidity_sensor'
    outside_air_relative_humidity_sensor_normalizer/set_observation_normalization_constants.sample_mean = 71.799372
    outside_air_relative_humidity_sensor_normalizer/set_observation_normalization_constants.sample_variance = 172.388773

    # measurement 19 outside_air_specificenthalpy_sensor
    outside_air_specificenthalpy_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_specificenthalpy_sensor'
    outside_air_specificenthalpy_sensor_normalizer/set_observation_normalization_constants.sample_mean = 60711.656343
    outside_air_specificenthalpy_sensor_normalizer/set_observation_normalization_constants.sample_variance = 25491060.173822

    # measurement 20 outside_air_temperature_sensor
    outside_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_temperature_sensor'
    outside_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 291.244931
    outside_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 12.904175

    # measurement 21 outside_air_wetbulb_temperature_sensor
    outside_air_wetbulb_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'outside_air_wetbulb_temperature_sensor'
    outside_air_wetbulb_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 287.709943
    outside_air_wetbulb_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 3.594260

    # measurement 22 program_differential_pressure_setpoint
    program_differential_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'program_differential_pressure_setpoint'
    program_differential_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 83808.578375
    program_differential_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 14897544.664858

    # measurement 23 program_supply_air_static_pressure_setpoint
    program_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'program_supply_air_static_pressure_setpoint'
    program_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 163.396282
    program_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 1092.073231

    # measurement 24 program_supply_air_temperature_setpoint
    program_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'program_supply_air_temperature_setpoint'
    program_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 289.490004
    program_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 2.854515

    # measurement 25 program_supply_water_temperature_setpoint
    program_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'program_supply_water_temperature_setpoint'
    program_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 341.467705
    program_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 74.961483

    # measurement 26 return_air_temperature_sensor
    return_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'return_air_temperature_sensor'
    return_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 295.602164
    return_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 11.309930

    # measurement 27 return_water_temperature_sensor
    return_water_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'return_water_temperature_sensor'
    return_water_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 326.219913
    return_water_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 497.847788

    # measurement 28 run_status
    run_status_normalizer/set_observation_normalization_constants.field_id = 'run_status'
    run_status_normalizer/set_observation_normalization_constants.sample_mean = -0.638340
    run_status_normalizer/set_observation_normalization_constants.sample_variance = 0.592523

    # measurement 29 speed_frequency_sensor
    speed_frequency_sensor_normalizer/set_observation_normalization_constants.field_id = 'speed_frequency_sensor'
    speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_mean = 7.003487
    speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_variance = 227.751249

    # measurement 30 speed_percentage_command
    speed_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'speed_percentage_command'
    speed_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 11.330966
    speed_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 602.718159

    # measurement 31 supervisor_supply_air_static_pressure_setpoint
    supervisor_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supervisor_supply_air_static_pressure_setpoint'
    supervisor_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 179.409052
    supervisor_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 352.049768

    # measurement 32 supervisor_supply_air_temperature_setpoint
    supervisor_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supervisor_supply_air_temperature_setpoint'
    supervisor_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 290.2
    supervisor_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 9.66245

    # measurement 33 supervisor_supply_water_temperature_setpoint
    supervisor_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supervisor_supply_water_temperature_setpoint'
    supervisor_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 332.164444
    supervisor_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 1.534112

    # measurement 34 supply_air_damper_percentage_command
    supply_air_damper_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'supply_air_damper_percentage_command'
    supply_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 51.173986
    supply_air_damper_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 1059.265742

    # measurement 35 supply_air_flowrate_sensor
    supply_air_flowrate_sensor_normalizer/set_observation_normalization_constants.field_id = 'supply_air_flowrate_sensor'
    supply_air_flowrate_sensor_normalizer/set_observation_normalization_constants.sample_mean = 177.520026
    supply_air_flowrate_sensor_normalizer/set_observation_normalization_constants.sample_variance = 50499.153481

    # measurement 36 supply_air_flowrate_setpoint
    supply_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supply_air_flowrate_setpoint'
    supply_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 209.557558
    supply_air_flowrate_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 47308.757207

    # measurement 37 supply_air_static_pressure_sensor
    supply_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.field_id = 'supply_air_static_pressure_sensor'
    supply_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.sample_mean = 128.527912
    supply_air_static_pressure_sensor_normalizer/set_observation_normalization_constants.sample_variance = 6679.599175

    # measurement 38 supply_air_static_pressure_setpoint
    supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supply_air_static_pressure_setpoint'
    supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 181.307432
    supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 361.757966

    # measurement 39 supply_air_temperature_sensor
    supply_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'supply_air_temperature_sensor'
    supply_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 289.737939
    supply_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 6.265837

    # measurement 40 supply_air_temperature_setpoint
    supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supply_air_temperature_setpoint'
    supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 289.329414
    supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 3.186769

    # measurement 41 supply_fan_run_status
    supply_fan_run_status_normalizer/set_observation_normalization_constants.field_id = 'supply_fan_run_status'
    supply_fan_run_status_normalizer/set_observation_normalization_constants.sample_mean = 0.439849
    supply_fan_run_status_normalizer/set_observation_normalization_constants.sample_variance = 0.806533

    # measurement 42 supply_fan_speed_frequency_sensor
    supply_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.field_id = 'supply_fan_speed_frequency_sensor'
    supply_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_mean = 15.926249
    supply_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants.sample_variance = 207.034194

    # measurement 43 supply_fan_speed_percentage_command
    supply_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.field_id = 'supply_fan_speed_percentage_command'
    supply_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.sample_mean = 26.543748
    supply_fan_speed_percentage_command_normalizer/set_observation_normalization_constants.sample_variance = 575.094979

    # measurement 44 supply_water_temperature_sensor
    supply_water_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'supply_water_temperature_sensor'
    supply_water_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 321.520315
    supply_water_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 658.413066

    # measurement 45 supply_water_temperature_setpoint
    supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'supply_water_temperature_setpoint'
    supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 320.261985
    supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 240.195517

    # measurement 46 zone_air_co2_concentration_sensor
    zone_air_co2_concentration_sensor_normalizer/set_observation_normalization_constants.field_id = 'zone_air_co2_concentration_sensor'
    zone_air_co2_concentration_sensor_normalizer/set_observation_normalization_constants.sample_mean = 432.092062
    zone_air_co2_concentration_sensor_normalizer/set_observation_normalization_constants.sample_variance = 962.903840

    # measurement 47 zone_air_co2_concentration_setpoint
    zone_air_co2_concentration_setpoint_normalizer/set_observation_normalization_constants.field_id = 'zone_air_co2_concentration_setpoint'
    zone_air_co2_concentration_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 739.337708
    zone_air_co2_concentration_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 3618.117781

    # measurement 48 zone_air_cooling_temperature_setpoint
    zone_air_cooling_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'zone_air_cooling_temperature_setpoint'
    zone_air_cooling_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 82.084227
    zone_air_cooling_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 402.158853

    # measurement 49 zone_air_heating_temperature_setpoint
    zone_air_heating_temperature_setpoint_normalizer/set_observation_normalization_constants.field_id = 'zone_air_heating_temperature_setpoint'
    zone_air_heating_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_mean = 64.231868
    zone_air_heating_temperature_setpoint_normalizer/set_observation_normalization_constants.sample_variance = 24.461668

    # measurement 50 zone_air_temperature_sensor
    zone_air_temperature_sensor_normalizer/set_observation_normalization_constants.field_id = 'zone_air_temperature_sensor'
    zone_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_mean = 190
    zone_air_temperature_sensor_normalizer/set_observation_normalization_constants.sample_variance = 408.113303

    supervisor_run_command_normalizer/set_observation_normalization_constants.field_id = 'supervisor_run_command'
    supervisor_run_command_normalizer/set_observation_normalization_constants.sample_mean = 0
    supervisor_run_command_normalizer/set_observation_normalization_constants.sample_variance = 1.0

    observation_normalizer_map = {{
    'building_air_static_pressure_sensor' : @building_air_static_pressure_sensor_normalizer/set_observation_normalization_constants(),
    'building_air_static_pressure_setpoint' : @building_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'cooling_percentage_command' : @cooling_percentage_command_normalizer/set_observation_normalization_constants(),
    'differential_pressure_sensor' : @differential_pressure_sensor_normalizer/set_observation_normalization_constants(),
    'differential_pressure_setpoint' : @differential_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'discharge_air_temperature_sensor' : @discharge_air_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'discharge_air_temperature_setpoint' : @discharge_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'exhaust_air_damper_percentage_command' : @exhaust_air_damper_percentage_command_normalizer/set_observation_normalization_constants(),
    'exhaust_air_damper_percentage_sensor' : @exhaust_air_damper_percentage_sensor_normalizer/set_observation_normalization_constants(),
    'exhaust_fan_speed_frequency_sensor' : @exhaust_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants(),
    'exhaust_fan_speed_percentage_command' : @exhaust_fan_speed_percentage_command_normalizer/set_observation_normalization_constants(),
    'heating_water_valve_percentage_command' : @heating_water_valve_percentage_command_normalizer/set_observation_normalization_constants(),
    'mixed_air_temperature_sensor' : @mixed_air_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'mixed_air_temperature_setpoint' : @mixed_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'outside_air_damper_percentage_command' : @outside_air_damper_percentage_command_normalizer/set_observation_normalization_constants(),
    'outside_air_dewpoint_temperature_sensor' : @outside_air_dewpoint_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'outside_air_flowrate_sensor' : @outside_air_flowrate_sensor_normalizer/set_observation_normalization_constants(),
    'outside_air_flowrate_setpoint' : @outside_air_flowrate_setpoint_normalizer/set_observation_normalization_constants(),
    'outside_air_relative_humidity_sensor' : @outside_air_relative_humidity_sensor_normalizer/set_observation_normalization_constants(),
    'outside_air_specificenthalpy_sensor' : @outside_air_specificenthalpy_sensor_normalizer/set_observation_normalization_constants(),
    'outside_air_temperature_sensor' : @outside_air_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'outside_air_wetbulb_temperature_sensor' : @outside_air_wetbulb_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'program_differential_pressure_setpoint' : @program_differential_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'program_supply_air_static_pressure_setpoint' : @program_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'program_supply_air_temperature_setpoint' : @program_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'program_supply_water_temperature_setpoint' : @program_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'return_air_temperature_sensor' : @return_air_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'return_water_temperature_sensor' : @return_water_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'run_status' : @run_status_normalizer/set_observation_normalization_constants(),
    'speed_frequency_sensor' : @speed_frequency_sensor_normalizer/set_observation_normalization_constants(),
    'speed_percentage_command' : @speed_percentage_command_normalizer/set_observation_normalization_constants(),
    'supervisor_supply_air_static_pressure_setpoint' : @supervisor_supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'supervisor_supply_air_temperature_setpoint' : @supervisor_supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'supervisor_supply_water_temperature_setpoint' : @supervisor_supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'supply_air_damper_percentage_command' : @supply_air_damper_percentage_command_normalizer/set_observation_normalization_constants(),
    'supply_air_flowrate_sensor' : @supply_air_flowrate_sensor_normalizer/set_observation_normalization_constants(),
    'supply_air_flowrate_setpoint' : @supply_air_flowrate_setpoint_normalizer/set_observation_normalization_constants(),
    'supply_air_static_pressure_sensor' : @supply_air_static_pressure_sensor_normalizer/set_observation_normalization_constants(),
    'supply_air_static_pressure_setpoint' : @supply_air_static_pressure_setpoint_normalizer/set_observation_normalization_constants(),
    'supply_air_temperature_sensor' : @supply_air_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'supply_air_temperature_setpoint' : @supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'supply_air_cooling_temperature_setpoint' : @supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'supply_air_heating_temperature_setpoint' : @supply_air_temperature_setpoint_normalizer/set_observation_normalization_constants(),

    'supply_fan_run_status' : @supply_fan_run_status_normalizer/set_observation_normalization_constants(),
    'supply_fan_speed_frequency_sensor' : @supply_fan_speed_frequency_sensor_normalizer/set_observation_normalization_constants(),
    'supply_fan_speed_percentage_command' : @supply_fan_speed_percentage_command_normalizer/set_observation_normalization_constants(),
    'supply_water_temperature_sensor' : @supply_water_temperature_sensor_normalizer/set_observation_normalization_constants(),
    'supply_water_setpoint' : @supply_water_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'zone_air_co2_concentration_sensor' : @zone_air_co2_concentration_sensor_normalizer/set_observation_normalization_constants(),
    'zone_air_co2_concentration_setpoint' : @zone_air_co2_concentration_setpoint_normalizer/set_observation_normalization_constants(),
    'zone_air_cooling_temperature_setpoint' : @zone_air_cooling_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'zone_air_heating_temperature_setpoint' : @zone_air_heating_temperature_setpoint_normalizer/set_observation_normalization_constants(),
    'zone_air_temperature_sensor' : @zone_air_temperature_sensor_normalizer/set_observation_normalization_constants(),

    'cooling_request_count': @request_count_observation_normalizer/set_observation_normalization_constants(),
    }}



    StandardScoreObservationNormalizer:
        normalization_constants = %observation_normalizer_map



    ##########################
    ### ENVIRONMENT
    ##########################


    # metrics_path = '/cns/oz-d/home/smart-buildings-control-team/smart-buildings/geometric_simulation_collects'
    discount_factor = 0.9

    num_days_in_episode=21
    metrics_reporting_interval=10
    label='2_cv_flr_1_outlined_floorplan_based_simulation_no_HVAC'
    num_hod_features = 12
    num_dow_features = 12

    Environment.building = @SimulatorBuilding()
    Environment.reward_function = @SetpointEnergyCarbonRegretFunction()
    Environment.observation_normalizer = @StandardScoreObservationNormalizer()
    Environment.action_config = @ActionConfig()
    Environment.metrics_reporting_interval = %metrics_reporting_interval

    Environment.discount_factor = %discount_factor
    Environment.label = %label
    Environment.num_days_in_episode= %num_days_in_episode
    Environment.default_actions = %default_actions
    Environment.num_hod_features = %num_hod_features
    Environment.num_dow_features = %num_dow_features


    HybridActionEnvironment.building = @SimulatorBuilding()
    HybridActionEnvironment.reward_function = @SetpointEnergyCarbonRegretFunction()
    HybridActionEnvironment.observation_normalizer = @StandardScoreObservationNormalizer()
    HybridActionEnvironment.action_config = @ActionConfig()
    HybridActionEnvironment.metrics_reporting_interval = %metrics_reporting_interval

    HybridActionEnvironment.discount_factor = %discount_factor
    HybridActionEnvironment.label = %label
    HybridActionEnvironment.num_days_in_episode= %num_days_in_episode
    HybridActionEnvironment.default_actions = %default_actions
    HybridActionEnvironment.num_hod_features = %num_hod_features
    HybridActionEnvironment.num_dow_features = %num_dow_features
    HybridActionEnvironment.device_action_tuples = [
        # --- AHU SYSTEM ACTIONS ---
        ('ahu', 'ahu_1_supervisor_run_command'),
        ('ahu', 'ahu_2_supervisor_run_command'),
        ('ahu', 'ahu_1_supply_air_temperature_setpoint'),
        ('ahu', 'ahu_1_static_pressure_setpoint'),
        ('ahu', 'ahu_2_supply_air_temperature_setpoint'),
        ('ahu', 'ahu_2_static_pressure_setpoint'),

        # --- HOT WATER SYSTEM ACTIONS ---
        ('hws', 'supervisor_run_command'),
        ('hws', 'supply_water_setpoint'),
        ('hws', 'differential_pressure'),
    ]
    """
  return gin_config_contents

In [ ]:
# @title Simulation Settings

# @markdown ### Viewing Ranges
vmin = 292 # @param {type:"integer"}
vmax = 296 # @param {type:"integer"}

# @markdown ---
# @markdown ### Outside Air Temperature
high_temp = 305 # @param {type:"number"}
low_temp = 305 # @param {type:"number"}

# @markdown ---
# @markdown ### HVAC State (Normalized Actions)

# @markdown **AHU 1**
ahu_1_run_command = 0 # @param [0, 1] {type:"raw"}
ahu_1_supply_air_temperature_setpoint = -1 # @param {type:"slider", min:-1, max:1, step:0.1}
ahu_1_static_pressure_setpoint = -1 # @param {type:"slider", min:-1, max:1, step:0.1}

# @markdown **AHU 2**
ahu_2_run_command = 1 # @param [0, 1] {type:"raw"}
ahu_2_supply_air_temperature_setpoint = -1 # @param {type:"slider", min:-1, max:1, step:0.1}
ahu_2_static_pressure_setpoint = 1 # @param {type:"slider", min:-1, max:1, step:0.1}

# @markdown **Hot Water System**
hws_run_command = 0 # @param [0, 1] {type:"raw"}
supply_water_setpoint = 1.0 # @param {type:"slider", min:-1, max:1, step:0.1}
differential_pressure = -1 # @param {type:"slider", min:-1, max:1, step:0.1}

GIN_CONFIG_PATH = "demo_config.gin"
with open(GIN_CONFIG_PATH, "w") as text_file:
  text_file.write(get_config(high_temp,low_temp))
gin.parse_config_file(GIN_CONFIG_PATH)

env = hybrid_action_environment.HybridActionEnvironment()
env.reset()
building_layout = env.building.simulator.building.floor_plan
renderer = building_renderer.BuildingRenderer(building_layout, 1)
def render_env(env):
  temps = env.building.simulator.building.temp
  image = renderer.render(temps, cmap='bwr',vmin=vmin,vmax=vmax).convert('RGB')
  return image

info_to_log = []
for i in range(200):

  action = {
      'discrete_action' : [ahu_1_run_command,ahu_2_run_command,hws_run_command], # ah1, ah2, hws
      'continuous_action' : [ahu_1_supply_air_temperature_setpoint,ahu_1_static_pressure_setpoint,ahu_2_supply_air_temperature_setpoint,
                             ahu_2_static_pressure_setpoint,supply_water_setpoint,differential_pressure],
  }
  reward = env.step(action) # take random action to step env
  timestamp = env.building.current_timestamp
  display(render_env(env))

  outside_air = env.building.simulator._hvac.air_handler._ahus[0].outside_air_temperature_sensor
  print(f'outside_air: {outside_air}')

  vavs = env.building.simulator._hvac._vavs.items()
  temps=[]
  dampers = []
  flow_rates=[]
  for v in vavs:
    temps.append(v[1].zone_air_temperature)
    dampers.append(v[1].damper_setting)
    flow_rates.append(v[1].flow_rate_demand)
    temp_window = v[1].thermostat.get_setpoint_schedule().get_temperature_window(timestamp)
  print("average temps:",sum(temps)/len(temps))
  print("average damper setting:",sum(dampers)/len(dampers))
  print("average vav flowrate:",sum(flow_rates)/len(flow_rates))
  info_to_log.append((temps[:13] + temps[13+1:],temp_window,outside_air,timestamp))
  plot_hvac_data_with_timestamps(info_to_log)
